# Hyperliquid × Bitcoin Fear/Greed Index
### Trader Behavior & Performance Analysis - How Market Sentiment Drives Trader Behavior on Hyperliquid

## Data Extraction

**Goal:** Load both raw datasets, parse and align timestamps, perform the
sentiment join, and engineer all key metrics needed for downstream analysis.
Output one clean working file: `working_trades.csv`.

**What this notebook produces:**
```
Load raw CSVs → Parse timestamps → Merge on date
→ Engineer metrics → Validate join → Export working_trades.csv
```

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

### Load Raw Data

In [2]:
fg_raw = pd.read_csv('fear_greed_index.csv')
tr_raw = pd.read_csv('historical_data.csv')
print('Fear/Greed shape:', fg_raw.shape)
print('Trader shape:', tr_raw.shape)

Fear/Greed shape: (2644, 4)
Trader shape: (211224, 16)


### Parse Timestamps

In [3]:
# Fear/Greed: date column is already YYYY-MM-DD
fg_raw['date'] = pd.to_datetime(fg_raw['date']).dt.date

# Trader data: Timestamp IST is DD-MM-YYYY HH:MM (IST timezone, daily join is fine)
tr_raw['date'] = pd.to_datetime(
    tr_raw['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce'
).dt.date

print('Null dates in trader data after parsing:', tr_raw['date'].isnull().sum())
print('Trader date range:', tr_raw['date'].min(), 'to', tr_raw['date'].max())

Null dates in trader data after parsing: 0
Trader date range: 2023-05-01 to 2025-05-01


### Prepare Fear/Greed for Join

Retain only the columns needed for analysis. The `timestamp` unix column is
redundant with `date` and dropped here.

In [4]:
fg = fg_raw[['date', 'value', 'classification']].copy()
fg.columns = ['date', 'fg_value', 'sentiment']

# Add binary sentiment label for simplified Fear vs Greed contrast
sentiment_map = {
    'Extreme Fear': 'Fear',
    'Fear': 'Fear',
    'Neutral': 'Neutral',
    'Greed': 'Greed',
    'Extreme Greed': 'Greed'
}
fg['sentiment_binary'] = fg['sentiment'].map(sentiment_map)
print('Fear/Greed table prepared:')
fg.head(3)

Fear/Greed table prepared:


,date,fg_value,sentiment,sentiment_binary
0,2018-02-01,30,Fear,Fear
1,2018-02-02,15,Extreme Fear,Fear
2,2018-02-03,40,Fear,Fear


### Select Working Columns from Trader Data

We retain only the columns needed for analysis. `Transaction Hash`, `Order ID`,
`Trade ID`, and raw `Timestamp` (unix ms) are tracking/audit fields — dropped here.

In [5]:
keep_cols = [
    'Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD',
    'Side', 'date', 'Start Position', 'Direction', 'Closed PnL',
    'Crossed', 'Fee'
]
tr = tr_raw[keep_cols].copy()
print('Working columns selected:', tr.columns.tolist())
print('Shape:', tr.shape)

Working columns selected: ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'date', 'Start Position', 'Direction', 'Closed PnL', 'Crossed', 'Fee']
Shape: (211224, 12)


### Merge Datasets on Date

In [6]:
merged = tr.merge(fg, on='date', how='left')
print('Rows after merge:', len(merged))
print('Null sentiment rows:', merged['sentiment'].isnull().sum())
print()
print('Sentiment distribution in merged data:')
print(merged['sentiment'].value_counts())

Rows after merge: 211224
Null sentiment rows: 6

Sentiment distribution in merged data:
sentiment
Fear             61837
Greed            50303
Extreme Greed    39992
Neutral          37686
Extreme Fear     21400
Name: count, dtype: int64


### Engineer Key Metrics

All metrics defined in Notebook 01 are computed here at the trade level.
Daily aggregations happen in the EDA and Deep Analysis notebooks.

In [7]:
# 1. Net PnL (after fees)
merged['net_pnl'] = merged['Closed PnL'] - merged['Fee']

# 2. Is a closing trade (PnL is only meaningful here)
merged['is_close'] = merged['Direction'].isin(['Close Long', 'Close Short'])

# 3. Is a winning trade (only on closing trades)
merged['is_win'] = (merged['Closed PnL'] > 0) & merged['is_close']

# 4. Direction category — simplified
direction_map = {
    'Open Long': 'open_long',
    'Close Long': 'close_long',
    'Open Short': 'open_short',
    'Close Short': 'close_short',
    'Buy': 'spot_buy',
    'Sell': 'spot_sell',
    'Long > Short': 'flip',
    'Short > Long': 'flip',
    'Spot Dust Conversion': 'dust',
    'Auto-Deleveraging': 'adl'
}
merged['direction_clean'] = merged['Direction'].map(direction_map)

# 5. Leverage proxy: Size USD / |Start Position| for trades where Start Position > 0
# Interpret: how large is this trade relative to existing position?
# Cap at 100x to handle edge cases
merged['leverage_proxy'] = np.where(
    merged['Start Position'].abs() > 0,
    (merged['Size USD'] / merged['Start Position'].abs()).clip(upper=100),
    np.nan  # undefined when starting from flat
)

# 6. Trade size bucket
merged['size_bucket'] = pd.cut(
    merged['Size USD'],
    bins=[0, 500, 2000, 10000, 1e9],
    labels=['small', 'medium', 'large', 'whale']
)

print('Metrics engineered. New columns added:')
print([c for c in merged.columns if c not in tr.columns and c not in fg.columns])

Metrics engineered. New columns added:
['net_pnl', 'is_close', 'is_win', 'direction_clean', 'leverage_proxy', 'size_bucket']


### Validate Engineered Metrics

In [8]:
print('Close trades:', merged['is_close'].sum())
print('Win trades:', merged['is_win'].sum())
print('Overall win rate (close trades):', round(merged['is_win'].sum() / merged['is_close'].sum() * 100, 1), '%')
print()
print('Net PnL describe:')
print(merged['net_pnl'].describe())
print()
print('Leverage proxy describe (where defined):')
print(merged['leverage_proxy'].dropna().describe())
print()
print('Size bucket distribution:')
print(merged['size_bucket'].value_counts())

Close trades: 84691
Win trades: 70754
Overall win rate (close trades): 83.5 %

Net PnL describe:
count    211224.000000
mean         47.585034
std         918.621638
min     -118071.556516
25%          -0.194153
50%          -0.005857
75%           5.535112
max      135299.803088
Name: net_pnl, dtype: float64

Leverage proxy describe (where defined):
count    207139.000000
mean         15.798022
std          34.080235
min           0.000000
25%           0.016545
50%           0.180349
75%           3.102770
max         100.000000
Name: leverage_proxy, dtype: float64

Size bucket distribution:
size_bucket
small     90832
medium    64501
large     37157
whale     18691
Name: count, dtype: int64


In [9]:
print('Direction clean distribution:')
merged['direction_clean'].value_counts()

Direction clean distribution:


direction_clean
open_long      49895
close_long     48678
open_short     39741
close_short    36013
spot_sell      19902
spot_buy       16716
dust             142
flip             127
adl                8
Name: count, dtype: int64

### Final Schema Check

In [10]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211224 entries, 0 to 211223
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype   
---  ------            --------------   -----   
 0   Account           211224 non-null  object  
 1   Coin              211224 non-null  object  
 2   Execution Price   211224 non-null  float64 
 3   Size Tokens       211224 non-null  float64 
 4   Size USD          211224 non-null  float64 
 5   Side              211224 non-null  object  
 6   date              211224 non-null  object  
 7   Start Position    211224 non-null  float64 
 8   Direction         211224 non-null  object  
 9   Closed PnL        211224 non-null  float64 
 10  Crossed           211224 non-null  bool    
 11  Fee               211224 non-null  float64 
 12  fg_value          211218 non-null  float64 
 13  sentiment         211218 non-null  object  
 14  sentiment_binary  211218 non-null  object  
 15  net_pnl           211224 non-null  float64 
 16  is

In [11]:
merged.head(3)

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,date,Start Position,Direction,Closed PnL,...,Fee,fg_value,sentiment,sentiment_binary,net_pnl,is_close,is_win,direction_clean,leverage_proxy,size_bucket
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,2024-12-02,0.000000,Buy,0.0,...,0.345404,80.0,Extreme Greed,Greed,-0.345404,False,False,spot_buy,NaN,large
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,2024-12-02,986.524596,Buy,0.0,...,0.005600,80.0,Extreme Greed,Greed,-0.005600,False,False,spot_buy,0.129424,small
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,2024-12-02,1002.518996,Buy,0.0,...,0.050431,80.0,Extreme Greed,Greed,-0.050431,False,False,spot_buy,1.147739,medium


### Export Working Dataset

In [12]:
merged.to_csv('working_trades.csv', index=False)
print('Exported working_trades.csv')
print('Shape:', merged.shape)
print('Columns:', merged.columns.tolist())

Exported working_trades.csv
Shape: (211224, 21)
Columns: ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'date', 'Start Position', 'Direction', 'Closed PnL', 'Crossed', 'Fee', 'fg_value', 'sentiment', 'sentiment_binary', 'net_pnl', 'is_close', 'is_win', 'direction_clean', 'leverage_proxy', 'size_bucket']


## Extraction Summary

| Step | Action | Result |
|---|---|---|
| Load | Both CSVs loaded | 2,644 + 211224 rows |
| Parse dates | Timestamp IST → date | 0 nulls |
| Merge | Left join on date | 99.994% match rate |
| Columns selected | 12 of 16 trader columns | Audit cols dropped |
| Metrics added | 6 engineered columns | net_pnl, is_close, is_win, leverage_proxy, size_bucket, direction_clean |
| Output | working_trades.csv | 211224 rows, 21 columns |

**Next:** Notebook 03 — Data Cleaning.